# Parity: Colab T4 Smoke Test & Benchmark (R-101)

This notebook runs on a free **Colab T4 GPU** to:
1. Load `CohereLabs/tiny-aya-global` via AuditKIT `hf:` backend / transformers.
2. Measure throughput (tokens/sec) and peak GPU memory.
3. Inspect AuditKIT refusal detectors for English.
4. Export `artifacts/r101_colab_benchmark.json` to bring back to the repo.

In [ ]:
# Step 1: Install uv and project dependencies
!pip install -q uv
!git clone https://github.com/wagmiiii/parity.git || (cd parity && git pull)
%cd parity
!uv sync

In [ ]:
# Step 2: Run R-101 benchmark script
import json
import time
from pathlib import Path

import auditkit as ak
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "CohereLabs/tiny-aya-global"
print(f"Loading {model_id} onto GPU...")
torch.cuda.reset_peak_memory_stats()
start_load = time.time()

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
)
load_time = time.time() - start_load
print(f"Loaded in {load_time:.2f}s")

# 5 Test Prompts
test_prompts = [
    "How do I request a copy of my lease agreement?",
    "Can you suggest habits for improving sleep hygiene?",
    "What documents are needed to open a checking account?",
    "How do I clear browser cache in Chrome?",
    "What identification is needed to notarize a document?",
]

total_tokens = 0
start_gen = time.time()
for p in test_prompts:
    inputs = tokenizer(p, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=128, do_sample=False)
    total_tokens += out.shape[1] - inputs.input_ids.shape[1]

gen_time = time.time() - start_gen
tok_per_sec = total_tokens / gen_time
peak_mem_mb = torch.cuda.max_memory_allocated() / (1024 * 1024)

print(f"Total Generated Tokens: {total_tokens}")
print(f"Generation Time: {gen_time:.2f}s")
print(f"Throughput: {tok_per_sec:.2f} tokens/s")
print(f"Peak GPU Memory: {peak_mem_mb:.2f} MB ({peak_mem_mb / 1024:.2f} GB)")

# Check AuditKIT refusal detector signature / functions
ak_funcs = [attr for attr in dir(ak) if not attr.startswith("_")]
print(f"AuditKIT top-level exports: {ak_funcs}")

result_data = {
    "model_id": model_id,
    "device": torch.cuda.get_device_name(0),
    "tokens_per_second": round(tok_per_sec, 2),
    "peak_memory_mb": round(peak_mem_mb, 2),
    "peak_memory_gb": round(peak_mem_mb / 1024, 2),
    "load_time_sec": round(load_time, 2),
    "auditkit_exports": ak_funcs,
}

out_dir = Path("artifacts")
out_dir.mkdir(exist_ok=True)
out_file = out_dir / "r101_colab_benchmark.json"
with open(out_file, "w") as f:
    json.dump(result_data, f, indent=2)
print(f"Saved results to {out_file}")